# EkaQuant Evaluation Notebook
This notebook evaluates **Qwen2.5-3B-Instruct** using the **eka-eval** framework and prepares the environment for **EkaQuant** task-aware selective quantization experiments.

In [ ]:
import os
from datetime import datetime
from kaggle_secrets import UserSecretsClient

# 1. Hugging Face Authentication
# Ensure you have a Kaggle Secret named 'HF_TOKEN'
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
    else:
        print("Warning: HF_TOKEN not found in Kaggle Secrets.")
except Exception as e:
    print(f"Could not authenticate with HF_TOKEN: {e}")

In [ ]:
# 2. Install Dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers bitsandbytes accelerate peft 
!pip install -q numpy scipy kneed scikit-image tqdm

# 3. Install eka-eval from IIT Gandhinagar
!git clone https://github.com/lingo-iitgn/eka-eval.git
%cd eka-eval
!pip install -q -r requirements-gpu.txt
!pip install -e .
%cd ..

# 4. Install EkaQuant
!git clone https://github.com/TarunNagarajan/EkaQuant.git
%cd EkaQuant
!pip install -e .
%cd ..

In [ ]:
# 5. Run Evaluation Baselines
model_id = "Qwen/Qwen2.5-3B-Instruct"
model_name = model_id.split("/")[-1]

# 8-bit Evaluation
print(f"Starting 8-bit evaluation for {model_id}...")
# eka-eval hardcodes 4bit in its model_loader.py; we patch it to 8bit for this run
!sed -i 's/load_in_4bit *= *True/load_in_8bit=True/g' eka-eval/eka_eval/core/model_loader.py
# eka-eval has a typo in its config for MMLU-IN; we patch the module path
!sed -i 's/indic\.mmlu_in\.evaluate_mmlu_in/multilingual.mmlu_in.evaluate_mmlu_in/g' eka-eval/eka_eval/config/benchmark_config.py
# Pipe responses to handle interactive prompts: [Local, HF, Model ID, No custom BM, Group 9 (INDIC), Task 1 (MMLU-IN), No visualization]
!printf "1\n1\n{model_id}\nno\n9\n1\nno\n" | python eka-eval/scripts/run_benchmarks.py --results_dir results_8bit_{model_name}

# 4-bit Evaluation
print(f"Starting 4-bit evaluation for {model_id}...")
# Reverting the patch back to 4bit
!sed -i 's/load_in_8bit *= *True/load_in_4bit=True/g' eka-eval/eka_eval/core/model_loader.py
!printf "1\n1\n{model_id}\nno\n9\n1\nno\n" | python eka-eval/scripts/run_benchmarks.py --results_dir results_4bit_{model_name}

In [ ]:
# 6. Summary of Results
import glob
import pandas as pd

csv_files = glob.glob("./results_*/calculated.csv", recursive=True)
if not csv_files:
    print("No results found yet.")
for f in csv_files:
    print(f"\n--- {f} ---")
    df = pd.read_csv(f)
    display(df)